# M6 BM25·pgvector 비교와 검색 오류 검토

같은 평가 질문과 정답 문서 ID를 놓고 두 검색 결과를 비교한다. 전체 자료 검색과 질문의 자료 유형을 미리 지정한 검색은 따로 본다. 누락·상위 1위 오류는 실제 검색 순위와 정답 문서 제목을 같이 확인한다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.retrieval_comparison import build_analysis, write_outputs

ROOT = Path.cwd()
summary, questions, errors, source_counts, metadata = build_analysis(ROOT)
write_outputs(ROOT, summary, questions, errors, source_counts, metadata)
print(f"평가 질문 {metadata['questions']}개 / 문서 청크 {metadata['chunks']:,}개 / 문서 {metadata['unique_documents']:,}개")
print(f"고정 질문 파일 해시: {metadata['questions_sha256']}")
display(source_counts)

In [ ]:
# 전체 점수와 자료 유형별 점수를 같은 지표로 비교한다.
overall = summary[summary["source_type"] == "ALL"].copy()
display(overall[[
    "scope", "questions", "bm25_hit1", "pgvector_hit1", "pgvector_minus_bm25_hit1",
    "bm25_hit5", "pgvector_hit5", "pgvector_minus_bm25_hit5",
    "bm25_precision5", "pgvector_precision5",
    "bm25_mrr5", "pgvector_mrr5", "pgvector_minus_bm25_mrr5",
    "both_hit5", "bm25_only_hit5", "pgvector_only_hit5", "neither_hit5"
]])
display(summary[summary["source_type"] != "ALL"][[
    "scope", "source_type", "questions",
    "bm25_hit1", "pgvector_hit1", "bm25_hit5", "pgvector_hit5",
    "bm25_mrr5", "pgvector_mrr5"
]].reset_index(drop=True))

In [ ]:
# 전체 검색과 자료 유형 필터 검색에서 점수 차이를 그린다.
metrics = ["hit1", "hit5", "mrr5"]
labels = ["Hit@1", "Hit@5", "MRR@5"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for ax, scope, title in zip(axes, ["all_sources", "source_filtered"], ["All sources", "Source filtered"]):
    row = overall[overall["scope"] == scope].iloc[0]
    x = range(len(metrics))
    width = 0.34
    b = [row[f"bm25_{m}"] for m in metrics]
    p = [row[f"pgvector_{m}"] for m in metrics]
    bars_b = ax.bar([i-width/2 for i in x], b, width, label="BM25", color="#3567A5")
    bars_p = ax.bar([i+width/2 for i in x], p, width, label="pgvector", color="#42A58A")
    ax.set_title(title)
    ax.set_xticks(list(x), labels)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Question fraction / score")
    ax.grid(axis="y", alpha=0.25)
    ax.bar_label(bars_b, fmt="%.3f", padding=2, fontsize=8)
    ax.bar_label(bars_p, fmt="%.3f", padding=2, fontsize=8)
axes[0].legend(frameon=False)
fig.suptitle("BM25 vs pgvector on the same 21 questions")
fig.tight_layout()
figure_path = ROOT / "data/personal/figures/bm25_pgvector_metric_comparison.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
display(fig)
print(f"그림 저장: {figure_path.relative_to(ROOT)}")

In [ ]:
# 전체 검색에서 두 방법이 각각 맞힌 질문 집합을 확인한다.
all_questions = questions[questions["scope"] == "all_sources"].copy()
print("Hit@5 결과 묶음:", all_questions["paired_hit5"].value_counts().to_dict())
display(all_questions[all_questions["paired_hit5"] != "both"][[
    "qid", "source_type", "query", "gold_doc_ids",
    "bm25_rank", "pgvector_rank", "paired_hit5"
]].sort_values("qid").reset_index(drop=True))

In [ ]:
# 정답이 1위가 아니거나 상위 5개에서 빠진 질문을 원문 제목과 대조한다.
display(errors[[
    "qid", "source_type", "query", "gold_doc_ids", "gold_title",
    "bm25_rank", "bm25_top1_doc_id", "bm25_top1_title",
    "pgvector_rank", "pgvector_top1_doc_id", "pgvector_top1_title",
    "error_family", "evidence_note"
]].reset_index(drop=True))

## BM25·pgvector 비교와 오류 분류

- 전체 자료 Hit@5는 BM25와 pgvector 모두 16/21(0.762)이지만 같은 질문을 맞힌 것은 아니다. 두 방식이 함께 성공한 12개, BM25만 성공 4개, pgvector만 성공 4개, 둘 다 실패 1개였다.
- BM25는 Hit@1 0.619, MRR@5 0.656, pgvector는 Hit@1 0.524, MRR@5 0.613이었다. 이 고정 세트에서는 BM25가 첫 결과와 평균 순위가 조금 앞섰지만, 21문항의 표본 결과라 모든 검색어에서 우세하다는 뜻은 아니다.
- 자료 유형 필터를 적용하면 BM25 Hit@5는 1.0, pgvector는 0.762였다. 자료 유형을 미리 아는 조건과 전체 검색을 구분해 발표한다.
- 실패는 표현 차이, 자료 유형 혼합, 관련 근거가 여러 문서에 나뉜 경우, OCR/청크 분할 등으로 분류해 원인 후보를 남겼다. 실패 질문을 보고 정답 라벨이나 코퍼스를 사후 변경하면 전후 비교가 공정하지 않으므로 다음 실험은 질문·라벨을 고정한다.

### 지금 결과에서 얻은 점

- 이 프로젝트 질문 세트에서는 어휘 기반과 의미 기반 검색을 함께 쓰는 혼합 검색을 시험할 근거가 있다. 다만 혼합이 더 낫다고 단정하려면 동일 질문과 라벨로 별도 측정해야 한다.
- 개선 우선순위는 둘 다 놓친 질문 1개와 각 방식만 맞힌 8개의 실제 검색 결과를 직접 읽고 정한다. 점수 차이만으로 검색 실패 원인을 확정하지 않는다.
